In [1]:
import dspy
from typing import Literal

dspy.configure_cache(
    enable_disk_cache=False,
    enable_memory_cache=False,
)

class ClassifyGuidelineEditorialCommentary(dspy.Signature):
    """
        Classify a biomedical abstract into ONE of three categories:

            - "guideline"
            - "editorial"
            - "commentary"

        Definitions:

        1. "guideline"

        Use this label ONLY when the abstract describes formal guidance that:
            - Is issued or endorsed by an official organisation, professional society,
            government or public health authority, or a named multidisciplinary
            working group (e.g., IDSA, ILADS, CDC, NIH panels, national health
            agencies, specialist colleges), AND
            - Aims to provide explicit recommendations, rules, or standards for
            clinical practice, diagnosis, treatment, follow-up, or policy.

        Typical cues include terms like: guideline(s), practice guideline,
        clinical practice guideline, consensus statement, evidence-based
        recommendation, position statement, practice parameter, society
        recommendations, official guidance.

        The abstract should clearly indicate that it sets or revises formal
        recommendations, criteria, or algorithms, rather than simply commenting
        on them.

        Do NOT label as "guideline" if:
            - The abstract is a single-author or small-group opinion piece (even if
            it uses prescriptive language like "should" or "ought"), OR
            - It is mainly arguing for or against existing guidelines without
            presenting new official recommendations.

        2. "editorial"

        Use this label when the abstract is primarily an opinion or viewpoint
        article that:
            - Expresses a clear stance, argument, or advocacy position,
            - Often reflects the author’s or journal’s perspective on a topic,
            controversy, policy, or guideline,
            - May discuss PTLDS/CLD guidelines or evidence but does NOT itself
            issue formal recommendations on behalf of an official body.

        Typical cues: editorial, viewpoint, perspective, opinion, "we argue",
        "we contend", reflective or advocacy tone.

        3. "commentary"

        Use this label when the abstract provides an analytical or explanatory
        discussion that:
            - Interprets, critiques, or contextualises a study, guideline, or topic,
            - Summarises or evaluates existing evidence,
            - May highlight implications, limitations, or future directions,
            - Does NOT present new formal guidelines or official recommendations.

        Typical cues: commentary, invited commentary, discussion, analysis,
        "we discuss", "we examine", focus on explanation or critique rather than
        prescriptive guidance.

        Borderline decision rules:

        - If the abstract explicitly refers to issuing or updating clinical
            practice guidelines, consensus recommendations, or a position
            statement on behalf of a named organisation or working group,
            choose "guideline".

        - If the abstract is mainly persuasive or argumentative (supporting or
            attacking a guideline, policy, or stance) without being an official
            organisational product, choose "editorial".

        - If the abstract mainly explains or critiques studies, guidelines, or
            concepts in an analytical tone, without issuing official recommendations,
            choose "commentary".

        Task:

        Read the abstract and output:
            (1) the single best-fitting label ("guideline", "editorial", or "commentary").

    """

    abstract: str = dspy.InputField(
        desc="The Abstract text to classify into guideline/editorial/commentary."
    )
    author: str = dspy.InputField(
        desc="The Author(s) of the Abstract."
    )

    label: Literal[
        "guideline",
        "editorial",
        "commentary",
    ] = dspy.OutputField(desc="The classification label.")


In [2]:
import json
import pandas as pd
with open("../../results/domain_classification/ensembled_classification_results_original.json", "r") as f:
    data = json.load(f)["results"]
filtered_data  = [item for item in data if item["primary_design"] == "guideline_or_editorial_or_commentary"] 
print(len(filtered_data))

path = "../../datasets/classification_combined_df-8k-dataset-minus2025-abstracts.csv"
all_data = pd.read_csv(path)

# add author information to filtered_data
for item in filtered_data:
    abstract = item["abstract"]
    matching_rows = all_data[all_data["abstract"] == abstract]
    print(matching_rows.iloc()[0]["authors"])
    if not matching_rows.empty:
        item["author"] = matching_rows.iloc()[0]["authors"]
    else:
        print("empty")
        item["author"] = "Unknown"

394
"Sue OConnell"
"Susan OConnell"
A. Berthele
A. Harvey
A. Louw, S. Schmidt, K. Zimney, E. Puentedura
A. MacDONALD, Springer-Verlag, Berlin Heidelberg
A. Pachner
A. Spreer, M. Djukic, R. Nau, H. Eiffert
A. Steere
Alexander Gerber, David Groneberg
Allen C. Steere
Allen C. Steere, Elise E. Drouin, Lisa J. Glickstein
Anjali Kumar, Jennings Hernandez
Anna Salińska, Piotr Węgrzyn, Konstancja Węgrzyn, Agnieszka Góra, Marcin Wasilewski, Maciej Nowicki, Julia Skwara, Dawid Barański, Natalia Dąbrowska, Gustaw Laskowski
B. Milovanovic, T. Gligorijevic
Bożena Muraczyńska, Edyta Gałęziowska
British Infection Association
C. Perske
C. Tranchant
Chinmoy Bhate, Robert A. Schwartz
Chloe Nichols, Brenda Windemuth
Christelle Sordet
D. G.P.WormserM.
D. Hassler
D. Kirmizis, G. Efstratiadis
D. Sunitha, P. Anusri, M. Sudhakar
David R. Snydman
David R. Snydman, Linden Hu
Dita Smíšková, Zuzana Blechová
EM Massarotti
Eugene D. Shapiro
F. Cabello, M. Embers, Stuart A. Newman, H. Godfrey
G. Keyßer
G. Stanek
Gab

In [3]:
import json
import os

student_llm_string = "openai/gpt-5-mini"
screener_results_path = "../../classifier/domain_classification/editorial_classification_results_gpt_groundtruth.json"
results_path = "../../results/domain_classification/editorial_classification_results_gpt_original.json"
results_path_save = "../../results/domain_classification/editorial_classification_results_gpt_original.jsonl"
API_KEY = os.getenv("openrouter_api_key")
student_lm = dspy.LM(
    model=student_llm_string,       # e.g. "openrouter/google/gemini-2.0-flash-001"
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    # (plus any model_params like temperature, max_tokens, etc)
    # temperature=1.0, top_p=1.0, seed=42
    temperature=1.0, max_tokens = 350000,
)
dspy.configure(lm=student_lm)

In [4]:
domain_classifier = dspy.ChainOfThought(ClassifyGuidelineEditorialCommentary)
# domain_classifier.load(path=screener_results_path)
counter = 0
print(len(filtered_data))
for example in filtered_data:
    pred = domain_classifier(abstract=example["abstract"], author=example["author"])
    with open(results_path_save, "a") as f:
         f.write(json.dumps({
        "abstract": example["abstract"],
        "author": example["author"],
        "label": pred.label,

    }) + "\n")
    print(f"Processed {counter} abstracts", end='\r')
    counter += 1
with open(results_path_save, "r") as f:
    data = [json.loads(line) for line in f.readlines()]
with open(results_path, "w") as f:
    json.dump({"results": data}, f, indent=4)

394
